In [1]:
import pandas as pd
import numpy as np
from scipy.io.wavfile import write
from tqdm import tqdm
import matplotlib.pyplot as plt
from scipy.signal import spectrogram

# Read the EEG data from a CSV file
data = pd.read_csv('../data/eeg_data.csv', low_memory=False)  # Use low_memory=False to avoid dtype warnings
data['TimeStamp'] = pd.to_datetime(data['TimeStamp'])
data.set_index('TimeStamp', inplace=True)

# Sample 10% of the data continuously in time
total_rows = data.shape[0]
sample_size = int(0.10 * total_rows)
start_index = np.random.randint(0, total_rows - sample_size)
data_sample_continuous = data.iloc[start_index:start_index + sample_size].copy()

# Combine Delta columns into a single column for the sample
data_sample_continuous['Delta_combined'] = data_sample_continuous[['Delta_TP9', 'Delta_AF7', 'Delta_AF8', 'Delta_TP10']].sum(axis=1)

# Calculate the mean of the Delta combined column in minute intervals for the sample
minute_resampled_sample = data_sample_continuous.index.to_period('Min')
data_sample_continuous['minute_resampled'] = data_sample_continuous.groupby(minute_resampled_sample)['Delta_combined'].transform('mean')

# Define frequencies for C major scale (do-re-mi-fa-sol-la-ti-do)
scale = np.array([261.63, 293.66, 329.63, 349.23, 392.00, 440.00, 493.88, 523.25])  # Frequencies in Hz

# Map EEG data continuously to musical notes in the C major scale for the sample
mapped_indices_sample = np.interp(data_sample_continuous['minute_resampled'], 
                                   (data_sample_continuous['minute_resampled'].min(), data_sample_continuous['minute_resampled'].max()), 
                                   (0, len(scale) - 1))  # Map to scale indices

# Convert indices to actual frequency values
mapped_frequencies_sample = scale[np.clip(np.round(mapped_indices_sample).astype(int), 0, len(scale) - 1)]

# Dynamic adjustment of pitch and volume factors based on wave shape for the sample
wave_amplitude_sample = np.abs(np.sin(mapped_frequencies_sample / max(scale)))  # Normalize wave shape
pitch_factor_sample = wave_amplitude_sample / np.max(wave_amplitude_sample)
volume_factor_sample = 0.5 + 0.5 * wave_amplitude_sample  # Adjust as needed for desired volume

# Convert volume_factor_sample to a NumPy array
volume_factor_sample = np.array(volume_factor_sample)

# Plot the mapped frequencies with dynamically adjusted volume factors for the sample
plt.figure(figsize=(12, 6))
plt.scatter(data_sample_continuous.index, mapped_frequencies_sample, c=volume_factor_sample, cmap='viridis', s=10)
plt.colorbar(label='Volume')
plt.plot(data_sample_continuous.index, mapped_frequencies_sample, color='gray', linestyle='-', linewidth=0.5)
plt.title('Mapped Frequencies with Dynamically Adjusted Volume (Sample)')
plt.xlabel('Time')
plt.ylabel('Frequency (Hz)')

# Save the plot as a PNG file
plt.savefig('scatter_mapped_frequencies_sample.png', format='png', dpi=300)  # Adjust dpi as needed
plt.close()  # Close the figure to free memory

# Generate synth audio data based on mapped frequencies of the sample
duration_sample = 30  # Duration in seconds for sample
sample_rate = 44100  # Sample rate for audio
time_values_sample = np.linspace(0, duration_sample, sample_rate * duration_sample)

print("Generating synth audio based on C major scale with volume adjustments for the sample...")
audio_data_sample = np.zeros(sample_rate * duration_sample)
with tqdm(total=len(mapped_frequencies_sample), desc="Synth Audio Progress") as pbar:
    for i, freq in enumerate(mapped_frequencies_sample):
        audio_data_sample += volume_factor_sample[i] * np.sin(2 * np.pi * freq * time_values_sample)
        pbar.update(1)

# Normalize the synth audio data of the sample
audio_data_sample /= np.max(np.abs(audio_data_sample))

# Save the synth audio data of the sample as a WAV file
write('synth_sample.wav', sample_rate, audio_data_sample.astype(np.float32))
print("\nSynth audio file generated successfully!")

# Generate revised audio data based on mapped frequencies and volume adjustments
print("Generating wave mapping audio based on mapped frequencies and volume adjustments...")
revised_audio_data = np.zeros(sample_rate * duration_sample)

# Create the revised audio signal with progress bar
with tqdm(total=len(mapped_frequencies_sample), desc="Wave Mapping Audio Progress") as pbar:
    for i, freq in enumerate(mapped_frequencies_sample):
        start_time = int(i * (duration_sample / len(mapped_frequencies_sample)) * sample_rate)
        t = np.linspace(0, duration_sample / len(mapped_frequencies_sample), int(sample_rate * (duration_sample / len(mapped_frequencies_sample))), endpoint=False)
        revised_audio_data[start_time:start_time + len(t)] += volume_factor_sample[i] * np.sin(2 * np.pi * freq * t)
        pbar.update(1)

# Normalize the revised audio data
revised_audio_data /= np.max(np.abs(revised_audio_data))

# Save the revised audio data of the sample as a WAV file
write('sine_waves_sample.wav', sample_rate, revised_audio_data.astype(np.float32))
print("\nWave mapping audio file generated successfully!")

# Generate ECG-like heart sound based on revised audio data
print("Generating ECG-like heart sound based on revised audio data...")
ecg_heartbeat_audio_data = np.zeros(sample_rate * duration_sample)

# Create the ECG-like heartbeat audio signal with progress bar
with tqdm(total=len(mapped_frequencies_sample), desc="ECG Heartbeat Audio Progress") as pbar:
    for i, freq in enumerate(mapped_frequencies_sample):
        start_time = int(i * (duration_sample / len(mapped_frequencies_sample)) * sample_rate)
        t = np.linspace(0, duration_sample / len(mapped_frequencies_sample), 
                       int(sample_rate * (duration_sample / len(mapped_frequencies_sample))), endpoint=False)

        # Lower frequency for a bass-like sound
        base_freq = freq * 0.6  # Lower frequency to simulate heartbeat
        heartbeat_freq = 1.0  # Heartbeat pulse frequency (1 Hz for 60 BPM)

        # Create a square wave for the heartbeat effect
        pulse = 0.5 * (1 + np.sign(np.sin(2 * np.pi * heartbeat_freq * t)))  # Square wave pulse

        # Combine the base frequency and the pulse for the heartbeat effect
        ecg_heartbeat_audio_data[start_time:start_time + len(t)] += (
            pulse * 0.5 * np.sin(2 * np.pi * base_freq * t)  # Multiply pulse with the sine wave
        )
        pbar.update(1)

# Normalize the ECG heartbeat audio data
max_value = np.max(np.abs(ecg_heartbeat_audio_data))
if max_value > 0:
    ecg_heartbeat_audio_data /= max_value

# Save the ECG heartbeat audio data of the sample as a WAV file
write('ecg_heartbeat_sample.wav', sample_rate, ecg_heartbeat_audio_data.astype(np.float32))
print("\nECG heartbeat audio file generated successfully!")

# Generate spectrograms for all audio files
def generate_spectrogram(audio_data, filename):
    frequencies, times, Sxx = spectrogram(audio_data, sample_rate)
    plt.figure(figsize=(12, 6))
    plt.pcolormesh(times, frequencies, 10 * np.log10(Sxx), shading='gouraud')
    plt.colorbar(label='Intensity (dB)')
    plt.title('Spectrogram of Audio Sound')
    plt.ylabel('Frequency (Hz)')
    plt.xlabel('Time (s)')
    plt.savefig(filename, format='png', dpi=300)
    plt.close()

# Generate spectrograms for all audio files
generate_spectrogram(audio_data_sample, 'spectrogram_synth_audio_sound.png')
generate_spectrogram(revised_audio_data, 'spectrogram_wave_mapping_audio_sound.png')
generate_spectrogram(ecg_heartbeat_audio_data, 'spectrogram_ecg_heartbeat_audio_sound.png')

Generating synth audio based on C major scale with volume adjustments for the sample...


Synth Audio Progress: 100%|█████████████████| 1830/1830 [00:46<00:00, 38.96it/s]



Synth audio file generated successfully!
Generating wave mapping audio based on mapped frequencies and volume adjustments...


Wave Mapping Audio Progress: 100%|███████| 1830/1830 [00:00<00:00, 23640.65it/s]



Wave mapping audio file generated successfully!
Generating ECG-like heart sound based on revised audio data...


ECG Heartbeat Audio Progress: 100%|██████| 1830/1830 [00:00<00:00, 16795.31it/s]



ECG heartbeat audio file generated successfully!
